# Random Forest Model

In [2]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

In [3]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import plotly.express as px
import plotly.graph_objects as go

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

ruta = r'C:\Users\jthow\iCloudDrive\Documents\3_Maestria_Estadistica_UNINORTE\3_Tercer_Semestre\Machine_Learning\tornados.csv.zip'  
df = pd.read_csv(ruta) 
df['loss'] = df['loss'].replace(0, pd.NA)
df['loss'] = df['loss'].interpolate(method='linear')
df['mag'] = df['mag'].fillna(df['mag'].mean())
df.isnull().sum()

om              0
yr              0
mo              0
dy              0
date            0
time            0
tz              0
datetime_utc    0
st              0
stf             0
mag             0
inj             0
fat             0
loss            0
slat            0
slon            0
elat            0
elon            0
len             0
wid             0
ns              0
sn              0
f1              0
f2              0
f3              0
f4              0
fc              0
dtype: int64

In [4]:
# Crear la columna 'mortality' en el DataFrame original
df['mortality'] = df['fat'].apply(lambda x: 0 if x == 0 else 1)

# Renombrar el DataFrame a 'mortality_target'
mortality_target = df
import numpy as np

# Crear la columna 'mortality' con 0 si 'fat' es 0, y 1 si 'fat' es mayor que 0
df['mortality'] = np.where(df['fat'] == 0, 0, 1)

# Contar la cantidad de ceros y unos
print("Cantidad de ceros:", (df['mortality'] == 0).sum())
print("Cantidad de unos:", (df['mortality'] == 1).sum())

# Asignar el DataFrame modificado a 'tornados.target'
tornados_target = df

Cantidad de ceros: 67120
Cantidad de unos: 1573


In [5]:
from sklearn.model_selection import train_test_split
X = tornados_target[['om', 'yr', 'mo', 'dy', 'stf', 'mag', 'inj', 'loss', 'slat', 'slon', 'elat', 'elon', 'len', 'wid', 'ns', 'sn', 'f1', 'f2', 'f3', 'f4']]
y = df['mortality']
# Dividir los datos en conjunto de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=42)
forest = RandomForestClassifier(n_estimators=5, random_state=2)
forest.fit(X_train, y_train)

RandomForestClassifier(n_estimators=5, random_state=2)

In [6]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier

def visualize_forest(forest, X_train, y_train, feature_names=None):
    """Visualización garantizada de Random Forest para datos de tornados"""
    # 1. Verificación básica de inputs
    if not hasattr(forest, 'estimators_'):
        raise ValueError("El modelo proporcionado no es un Random Forest con árboles accesibles")
    
    # 2. Preparación de datos
    X = X_train.values if hasattr(X_train, 'values') else np.array(X_train)
    y = y_train.values if hasattr(y_train, 'values') else np.array(y_train)
    
    # 3. Selección automática de 2 características importantes
    if feature_names is None:
        feature_names = [f"Feature {i}" for i in range(X.shape[1])]
    
    if X.shape[1] < 2:
        raise ValueError("Se necesitan al menos 2 características para la visualización")
    
    # Seleccionar las 2 características más importantes
    if hasattr(forest, 'feature_importances_'):
        top_features = np.argsort(forest.feature_importances_)[-2:]
    else:
        top_features = [0, 1]  # Por defecto
    
    X_2d = X[:, top_features]
    selected_features = [feature_names[i] for i in top_features]

    # 4. Configuración de la figura
    plt.figure(figsize=(15, 8))
    
    # 5. Visualización del Random Forest completo
    try:
        from mlxtend.plotting import plot_decision_regions
        plot_decision_regions(X_2d, y, clf=forest, legend=2)
        plt.xlabel(selected_features[0])
        plt.ylabel(selected_features[1])
        plt.title("Fronteras de Decisión del Random Forest")
    except ImportError:
        # Fallback si no tienes mlxtend instalado
        plt.scatter(X_2d[:, 0], X_2d[:, 1], c=y, cmap='coolwarm', alpha=0.6)
        plt.xlabel(selected_features[0])
        plt.ylabel(selected_features[1])
        plt.title("Distribución de Clases (sin fronteras de decisión)")
    
    plt.show()
    
    # 6. Visualización de los primeros 4 árboles
    n_trees = min(4, len(forest.estimators_))
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    for i, ax in enumerate(axes.ravel()[:n_trees]):
        from sklearn.tree import plot_tree
        plot_tree(forest.estimators_[i], 
                 feature_names=feature_names,
                 class_names=['No VICTIMAS', 'VICTIMAS'],
                 filled=True, 
                 ax=ax)
        ax.set_title(f"Árbol {i+1}")
    
    plt.tight_layout()
    plt.show()

# Ejemplo de uso:
# visualize_forest(forest_model, X_train, y_train, feature_names=['mag', 'fat', 'len', 'wid'])

In [7]:
print("Dimensiones de X_train:", X_train.shape)
print("Clases en y_train:", np.unique(y_train))

Dimensiones de X_train: (51519, 20)
Clases en y_train: [0 1]


## Metricas Para el Modelo Random Forest

In [11]:
# ------------------------
# Paso 1: Importar los paquetes necesarios
# ------------------------
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_absolute_percentage_error, 
    mean_squared_error, 
    r2_score, 
    precision_score, 
    recall_score, 
    accuracy_score, 
    f1_score, 
    roc_auc_score
)
from statsmodels.stats.diagnostic import acorr_ljungbox
from scipy.stats import jarque_bera
from joblib import dump
from time import time  # Importar time para medir el tiempo de entrenamiento

# ------------------------
# Paso 2: Cargar los datos
# ------------------------
# Suponiendo que X_train, X_test, y_train, y_test ya están definidas

# ------------------------
# Paso 3: Definir el pipeline y el grid de hiperparámetros
# ------------------------
# Definir el pipeline para el modelo de RandomForestRegressor
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RandomForestRegressor(random_state=42))
])

# Definir el grid de hiperparámetros para RandomForestRegressor
param_grid = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [None, 10, 20],
    'model__min_samples_split': [2, 5, 10]
}

# ------------------------
# Paso 4: Entrenar el modelo con GridSearchCV
# ------------------------
# Medir el tiempo de entrenamiento
start_time = time()

# Realizar GridSearchCV para el modelo
grid_search = GridSearchCV(pipeline, param_grid, cv=5, n_jobs=-1, scoring='neg_mean_squared_error')
grid_search.fit(X_train, y_train)

# Calcular el tiempo total de entrenamiento
training_time = time() - start_time

# Obtener el mejor modelo
best_model = grid_search.best_estimator_

# Guardar el modelo entrenado
dump(grid_search, 'RandomForestRegressor.joblib')

# ------------------------
# Paso 5: Hacer predicciones con el mejor modelo
# ------------------------
y_pred = best_model.predict(X_test)
residuals = y_test - y_pred  # Calcular residuos

# ------------------------
# Paso 6: Calcular las métricas de regresión
# ------------------------
mape = mean_absolute_percentage_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)
jb_p_value = jarque_bera(residuals)[1]

# ------------------------
# Paso 7: Realizar el Ljung-Box test para comprobar autocorrelación en los residuos
# ------------------------
ljung_box_test = acorr_ljungbox(residuals, lags=[10], return_df=True)
ljung_box_stat = ljung_box_test['lb_stat'].iloc[-1]
ljung_box_pvalue = ljung_box_test['lb_pvalue'].iloc[-1]

# Imprimir el resultado del Ljung-Box test
print(f"Ljung-Box Test statistic: {ljung_box_stat:.4f}")
print(f"Ljung-Box Test p-value: {ljung_box_pvalue:.4f}")

# ------------------------
# Paso 8: Convertir el problema de regresión a clasificación
# ------------------------
# Usar la mediana de y_test como umbral
threshold = np.median(y_test)
y_test_class = (y_test > threshold).astype(int)
y_pred_class = (y_pred > threshold).astype(int)

# ------------------------
# Paso 9: Calcular las métricas de clasificación
# ------------------------
precision = precision_score(y_test_class, y_pred_class)
recall = recall_score(y_test_class, y_pred_class)
accuracy = accuracy_score(y_test_class, y_pred_class)
f1 = f1_score(y_test_class, y_pred_class)
auc = roc_auc_score(y_test_class, y_pred_class)

# ------------------------
# Paso 10: Crear DataFrame con los resultados
# ------------------------
resultados_rf = pd.DataFrame({
    'Modelo': ['RandomForestRegressor optimizado (GridSearchCV)'],
    'MAPE': [f"{mape:.2f}"],
    'RMSE': [f"{rmse:.2f}"],
    'R Cuadrado': [f"{r2:.2f}"],
    'Ljung-Box Test statistic': [f"{ljung_box_stat:.4f}"],
    'Ljung-Box Test p-value': [f"{ljung_box_pvalue:.4f}"],
    'Jarque-Bera p-value': [f"{jb_p_value:.2f}"],
    'Precision': [f"{precision:.2f}"],
    'Recall': [f"{recall:.2f}"],
    'Accuracy': [f"{accuracy:.2f}"],
    'F1-Score': [f"{f1:.2f}"],
    'AUC': [f"{auc:.2f}"],
    'CPU time (s)': [round(training_time, 2)]  # Incluir el tiempo de entrenamiento
})

# ------------------------
# Paso 11: Mostrar los resultados
# ------------------------
print("Métricas para el modelo de RandomForestRegressor:")
display(resultados_rf)

Ljung-Box Test statistic: 18.7850
Ljung-Box Test p-value: 0.0431
Métricas para el modelo de RandomForestRegressor:


,Modelo,MAPE,RMSE,R Cuadrado,Ljung-Box Test statistic,Ljung-Box Test p-value,Jarque-Bera p-value,Precision,Recall,Accuracy,F1-Score,AUC,CPU time (s)
0,RandomForestRegressor optimizado (GridSearchCV),63631017756016.79,0.12,0.36,18.7850,0.0431,0.00,0.02,1.00,0.02,0.04,0.50,853.61
